In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split

class TerrainClassifier1DCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(TerrainClassifier1DCNN, self).__init__()
        
        # Input shape per step: (Batch, 6 channels, 69 timesteps)
        self.features = nn.Sequential(
            nn.Conv1d(in_channels=6, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 69 -> 34
            
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 34 -> 17
            
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 17 -> 8
            
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # 8 -> 4
            
            nn.Dropout(p=0.5)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(128 * 4, 64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

def generate_tactile_dataset(num_samples=1600):
    # 4 classes: 0: High Friction, 1: Low Friction, 2: Deformable, 3: Granular
    y = np.random.randint(0, 4, size=(num_samples,)).astype(np.int64)
    X = np.zeros((num_samples, 6, 69), dtype=np.float32)
    
    for i in range(num_samples):
        label = y[i]
        # Baseline sensor noise & mechanical vibrations
        X[i] = np.random.randn(6, 69) * 0.8
        
        # Temporal jitter: foot strike onset varies dynamically across gait cycles
        start_idx = np.random.randint(5, 25)
        amp_var = np.random.uniform(0.6, 1.4)
        
        if label == 0:
            # High Friction: sharp normal impact + moderate-to-high shear force
            X[i, 0:5, start_idx:start_idx+12] += np.hanning(12) * 2.5 * amp_var
            X[i, 5, start_idx:start_idx+12] += np.hanning(12) * np.random.uniform(1.2, 2.2)
        elif label == 1:
            # Low Friction: sharp normal impact + low-to-moderate shear force (overlapping)
            X[i, 0:5, start_idx:start_idx+12] += np.hanning(12) * 2.5 * amp_var
            X[i, 5, start_idx:start_idx+12] += np.hanning(12) * np.random.uniform(0.4, 1.5)
        elif label == 2:
            # Deformable: prolonged ground contact duration, wider impact curve
            end_idx = min(start_idx + 28, 69)
            length = end_idx - start_idx
            X[i, 0:5, start_idx:end_idx] += np.hanning(length) * 1.8 * amp_var
            X[i, 5, start_idx:end_idx] += np.hanning(length) * 0.8 * amp_var
        elif label == 3:
            # Granular: erratic surface collapse with distributed tactile noise
            X[i] += np.random.randn(6, 69) * 0.9
            X[i, 0:6, start_idx:start_idx+10] += np.hanning(10) * 1.4 * amp_var
            
    return X, y

def main():
    print("Synthesizing realistic 6x69 tactile tensor dataset...")
    X_raw, y_raw = generate_tactile_dataset(num_samples=1600)
    
    # 80/20 train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
    )
    
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), 
        batch_size=32, 
        shuffle=True
    )
    test_loader = DataLoader(
        TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), 
        batch_size=32, 
        shuffle=False
    )
    
    # Hardware acceleration check (Apple Silicon MPS -> CUDA -> CPU)
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print(f"Executing model training on device: {device}\n")
    
    model = TerrainClassifier1DCNN(num_classes=4).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    epochs = 12
    for epoch in range(epochs):
        # Training Phase
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        # Validation Phase on Unseen Data
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        val_acc = 100.0 * correct / total
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] - Train Loss: {avg_loss:.4f} - Validation Accuracy: {val_acc:.2f}%")

if __name__ == "__main__":
    main()

Synthesizing realistic 6x69 tactile tensor dataset...
Executing model training on device: mps

Epoch [01/12] - Train Loss: 1.2594 - Validation Accuracy: 57.81%
Epoch [02/12] - Train Loss: 0.7905 - Validation Accuracy: 69.06%
Epoch [03/12] - Train Loss: 0.5377 - Validation Accuracy: 75.00%
Epoch [04/12] - Train Loss: 0.4426 - Validation Accuracy: 74.38%
Epoch [05/12] - Train Loss: 0.4269 - Validation Accuracy: 71.25%
Epoch [06/12] - Train Loss: 0.4078 - Validation Accuracy: 73.44%
Epoch [07/12] - Train Loss: 0.3860 - Validation Accuracy: 74.69%
Epoch [08/12] - Train Loss: 0.3750 - Validation Accuracy: 73.75%
Epoch [09/12] - Train Loss: 0.3664 - Validation Accuracy: 75.31%
Epoch [10/12] - Train Loss: 0.3855 - Validation Accuracy: 73.75%
Epoch [11/12] - Train Loss: 0.3616 - Validation Accuracy: 73.75%
Epoch [12/12] - Train Loss: 0.3603 - Validation Accuracy: 76.88%
